In [0]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import timeit

In [0]:
builder = (SparkSession.builder
           .appName("optimize-table-partitions-delta")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [0]:
# Create some sample data frames
# A large data frame with 1 million rows
large_df = (spark.range(0, 1000000)
            .withColumn("salary", 100*(rand() * 100).cast("int"))
            .withColumn("gender", when((rand() * 2).cast("int") == 0, "M").otherwise("F"))
            .withColumn("country_code", 
                        when((rand() * 4).cast("int") == 0, "US")
                        .when((rand() * 4).cast("int") == 1, "CN")
                        .when((rand() * 4).cast("int") == 2, "IN")
                        .when((rand() * 4).cast("int") == 3, "BR")
                        .otherwise('RU')))
large_df.show(5)

In [0]:
(large_df.write
 .format("delta")
 .mode("overwrite")
 .save("../data/tmp/large_delta"))

In [0]:
(large_df.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("country_code")
 .option("overwriteSchema", "true")
 .save("../data/tmp/large_delta_partitioned"))

In [0]:
non_partitioned_query = "spark.sql(\"SELECT country_code,gender, COUNT(*) AS employees FROM delta.`/opt/workspace/data/tmp/large_delta` GROUP BY ALL ORDER BY employees DESC\").show()"
non_partitioned_time = timeit.timeit(non_partitioned_query, number=1, globals=globals())
print(f"Non-partitioned query time: {non_partitioned_time} seconds")

In [0]:
partitioned_query = "spark.sql(\"SELECT country_code,gender, COUNT(*) AS employees FROM delta.`/opt/workspace/data/tmp/large_delta_partitioned` GROUP BY ALL ORDER BY employees DESC\").show()"
partitioned_time = timeit.timeit(partitioned_query, number=1, globals=globals())
print(f"Partitioned query time: {partitioned_time} seconds")


In [0]:
spark.stop()